# 09 · Watch choices become a trace

**Skill → choose a tool → execute → inspect result → choose again**

This lesson shows a separate teaching loop instrumented with Relay and routed through Switchyard. Its observation is a **recorded VSS description**; model requests are live.

**Who does what:** There are two layers: OpenClaw/Nano orchestrates the lesson’s three tools. Inside the run-loop tool, Lightning base controls the teaching loop and writes drafts; Lightning specialist classifies. Switchyard routes calls and Relay records them.

###  bounded agents and execution traces

Relay **0.9.4** [overview](https://docs.nvidia.com/nemo/relay/v0.9.4/about-nemo-relay/overview) and [managed tool calls](https://docs.nvidia.com/nemo/relay/v0.9.4/instrument-applications/instrument-tool-call), plus the [Switchyard guide](https://nvidia-nemo.github.io/Switchyard/getting_started/).

finish 08, provision Lightning, and execute tools → run → fail. The loop uses your skill, asks base for JSON actions, validates allowed actions, and calls specialist for classification. Relay records model/tool boundaries. No Relay daemon is needed for this in-process subscriber.

**Inspect the mechanism:** [09 lesson source](agent/lessons/09-agent-loop.json) defines `run`/`execute`; [routing_trace.py](agent/routing_trace.py) wraps SDK calls. The displayed video is optional context: `perceive` reads a recorded description. The failure cell deliberately substitutes a wrong action; its Relay error is real but its controller choice is injected.

**Framework boundary:** this is a custom Python loop. NAT 1.9.0 is used only by the browser demo. NVIDIA's [NAT migration notice](https://docs.nvidia.com/nemo/agent-toolkit/latest/resources/migration-guide.html) marks that package deprecated; do not treat the browser harness as a new-project framework recommendation. The outer OpenClaw/Nano handoff is another layer, not this loop's controller.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('09-agent-loop')

### 1 · See the tools and their order

| Tool | What it does |
|---|---|
| perceive | Read a recorded observation |
| triage | Ask the specialist and check its answer |
| draft | Ask base for a source-linked draft |
| request_review | Hold with a reason |
| finish | Stop |

No tool releases a report. The Python implementation stays behind the short cell, available in the original notebook.

In [ ]:
lesson.step("tools")
print("Allowed before evidence:", lesson.ns["allowed_actions"]({"clip":"civil-market","released":False}, "draft-report"))
print("Skill instructions:\n", lesson.ns["load_skill"]("draft-report")[0])

### 2 · Run and inspect the actual decisions

A possible sequence is **perceive → triage → draft → finish**. A HOLD can be the correct outcome when validation fails. The loop allows at most eight decisions.

The reason below is the model’s emitted explanation, not hidden reasoning.

**Example clip · Civilian market**

Play or scrub the video, then run the agent cell below. This is the clip used by both this example and the TUI task. The loop reads its saved VSS description; playback does not trigger fresh video analysis.

In [ ]:
from IPython.display import Video
display(Video(url="/files/run/chain/clips/civil-market.mp4", width=720,
              html_attributes='controls muted playsinline preload="metadata"'))

In [ ]:
lesson.step("run")
record = lesson.read("agent-run.json")
table(["Step", "Tool", "Stated reason", "Seconds", "Result"], [(e["step"], (e.get("action") or {}).get("tool"), (e.get("action") or {}).get("reason"), e["seconds"], e.get("error", "returned result")) for e in record["events"]])
print("Outcome:", record["status"])
display(JSON(record["state"], expanded=False))

### API pattern · let Relay execute the tool callback

Adapted from Relay's [managed tool-call guide](https://docs.nvidia.com/nemo/relay/v0.9.4/instrument-applications/instrument-tool-call) and our `Trace.tool` implementation. Reference only; the preceding run already captures these events.

```python
async def invoke(arguments):
    value = execute(arguments, state, skill, model_call)
    return relay.ToolExecutionResult(value)

result = await relay.tools.execute(
    action['tool'], action, invoke, handle=trace.handle,
)
```

`Trace` owns the scope, subscriber, flush, and JSON output. Inspect [routing_trace.py](agent/routing_trace.py) before adapting this pattern: the callback needs the run's state and model client. Relay records execution; it does not decide which action is allowed.


### 3 · Inspect the Relay events

Relay captures the model and tool lifecycle, including timing and errors. The rows below are derived from this run’s actual events.

In [ ]:
from course.agent.routing_trace import timeline
events = timeline(lesson.ns["trace"].events)
display(JSON(events, expanded=False))

### 4 · Inject one wrong action

**No observation → triage requested → rejected → Relay ERROR**

This is an intentional missing-evidence test, not a spontaneous model mistake. The real tool precondition still runs. We retain this trace for notebook 11.

In [ ]:
lesson.step("fail")
failed = lesson.read("failed-agent-run.json")
table(["Injected action", "Actual error"], [(failed["events"][0]["action"]["tool"], failed["events"][0]["error"])])
display(JSON(timeline(lesson.ns["failure_trace"].events), expanded=False))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-run-loop**. Open the actual instructions below.

**`inspect-skill` → `run-loop` → `check-run`**

This lesson also uses NVIDIA’s pinned **nemo-relay-instrument-calls** skill. OpenClaw must read both skills before executing tools.

In [ ]:
show_skill_library('09-agent-loop')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-run-loop/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-run-loop/
        │   └── SKILL.md
        ├── nemo-relay-instrument-calls/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder). It also tells the agent to read `nemo-relay-instrument-calls` before executing tools.

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `08-agent-skills`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('09-agent-loop')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-run-loop/SKILL.md")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/nemo-relay-instrument-calls/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-skill', 'run-loop', 'check-run']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. There are two layers: OpenClaw/Nano orchestrates the lesson’s three tools. Inside the run-loop tool, Lightning base controls the teaching loop and writes drafts; Lightning specialist classifies. Switchyard routes calls and Relay records them.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** perceive → triage → draft → finish, or request_review when a check fails.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** Keep the skill version, model routes, tool results, and failures together. A trace makes the next change testable.

Exit the TUI with `/exit` when done.